# 11 · Chọn cấu hình OCR trên dev300: backend × template (W8, bước 5)

**Vấn đề:** Notebook 10 cho thấy hai backend OCR mạnh yếu khác nhau trên dev300 (266 ảnh):
- `paddleocr` (PP-OCRv6, mô hình Latin): đọc đủ mặt chữ hơn nhưng **làm rơi dấu tiếng Việt**, ví dụ "Kiểu Thẩm" → "Kiu Thm". Đáp án gỡ được từ OCR chỉ 27,0% khi giữ dấu, nhưng lên 57,7% khi bỏ dấu.
- `paddledet_vietocr` (cùng bộ dò chữ, VietOCR đọc): **giữ dấu tốt hơn**, 37,7% giữ dấu và 48,7% bỏ dấu. McNemar p = 0,0008 so với `paddleocr` ở tiêu chí giữ dấu.

Chỉ số đó chỉ đo OCR **có đưa được đáp án vào prompt hay không**. Điều cần biết là **Qwen có dùng được OCR để trả lời đúng hơn không**: model có thể tự sửa dấu, hoặc ngược lại bị OCR sai dắt đi lệch. Notebook này đo trực tiếp điều đó, và chọn cấu hình theo **luật đã ghi trong `docs/W08/README.md` trước khi có số liệu**.

**Chạy 5 run, đều là B1 (zero-shot, cấu hình chốt W07) trên dev300, chỉ khác phần OCR trong prompt:**

| Run | OCR |
|---|---|
| B1 | không có (mốc so sánh) |
| B1 + paddleocr · v1 | dòng OCR trung tính |
| B1 + vietocr · v1 | dòng OCR trung tính |
| B1 + paddleocr · v2 | dòng OCR có báo trước "có thể sai chính tả hoặc thiếu dấu" |
| B1 + vietocr · v2 | như trên |

**Runtime: L4** (cùng GPU với B1/B2). Khoảng 35–45 phút, chạy được bằng **Run all**: notebook tự dừng ở cuối, không đụng tới test.

## 1. Thiết lập môi trường

Giống notebook 02/04/08/09 (cùng pin `transformers==5.17.0`). Dừng hẳn nếu GPU không phải L4.

*Kỳ vọng: commit mới nhất (có `src/eval/select_ocr_config.py`), `GPU: NVIDIA L4`, `✅ Thiết lập xong`.*

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, sys
if not os.path.exists('/content/ViVQA-VLM/.git'):
    !cd /content && git clone -q https://github.com/PLHGNAOH/ViVQA-VLM.git
%cd /content/ViVQA-VLM
!git pull -q origin main
!git log -1 --oneline
sys.path.insert(0, '/content/ViVQA-VLM')
assert os.path.exists('src/eval/select_ocr_config.py'), 'Code cũ -> git pull chưa lấy được commit W08 bước 5'

!pip install -q transformers==5.17.0 accelerate qwen-vl-utils bitsandbytes peft huggingface_hub pyyaml

try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')   # không in token ra
    print('HF_TOKEN: đã nạp')
except Exception:
    print('HF_TOKEN: không có (ổn nếu zip ảnh đã nằm trên Drive)')

import torch
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ GPU'
print('GPU:', GPU)
if 'L4' not in GPU:
    raise SystemExit('❌ Phải chạy trên L4 (cùng GPU với B1/B2). Runtime -> Change runtime type -> L4, rồi chạy lại cell này.')
print('✅ Thiết lập xong')

## 2. Tên run + kiểm tra trước khi tốn GPU

- Cấu hình model phải đúng bản chốt W07 (bf16, 512 token, 4-bit NF4).
- Hai OCR cache phải **đủ cả 266 ảnh** của dev300 (tạo ở notebook 10).
- File chỉ số `ocr_coverage.json` của notebook 10 phải có trên Drive.
- Chạy selftest của `run_baseline` và `select_ocr_config` (không cần GPU).

*Kỳ vọng: `bfloat16 | 401408 | nf4`, hai dòng `… 266/266 ảnh`, hai dòng `SELFTEST OK`.*

In [ ]:
import json, yaml
from src.data.ocr import default_cache_path, load_ocr_cache, images_of
from src.eval.run_baseline import prepare_images

DRIVE_ROOT = '/content/drive/MyDrive/ViVQA-VLM'
DS = 'vitextvqa_official'
EXP_ROOT = f'{DRIVE_ROOT}/experiments'
DEV = f'{DRIVE_ROOT}/data/{DS}/dev_subset300_seed42.json'
COVERAGE = f'{EXP_ROOT}/W08_ocr_dev300_coverage/ocr_coverage.json'
OUT_DIR = f'{EXP_ROOT}/W08_ocr_select_dev300_L4_seed42'
TAIL = 'qwen25vl_vitextvqa_dev300_bf16_L4_seed42'
B1 = f'W08_B1_zeroshot_{TAIL}'
RUNS = {('paddleocr', 'v1'): f'W08_B1ocr_paddleocr_v1_{TAIL}',
        ('paddledet_vietocr', 'v1'): f'W08_B1ocr_vietocr_v1_{TAIL}',
        ('paddleocr', 'v2'): f'W08_B1ocr_paddleocr_v2_{TAIL}',
        ('paddledet_vietocr', 'v2'): f'W08_B1ocr_vietocr_v2_{TAIL}'}

cfg = yaml.safe_load(open('configs/qwen_lora.yaml', encoding='utf-8'))
print(cfg['model']['torch_dtype'], '|', cfg['model']['max_pixels'], '|', cfg['quantization']['bnb_4bit_quant_type'])
IMG = prepare_images(DS, DRIVE_ROOT)
need = set(images_of([DEV], IMG))
for backend in ('paddleocr', 'paddledet_vietocr'):
    c = load_ocr_cache(default_cache_path(DRIVE_ROOT, DS, backend))
    have = len(need & set(c['images']))
    print(f'{backend:18s} {have}/{len(need)} ảnh | {c["engine_info"].get("rec_model")}')
    assert have == len(need), f'Cache {backend} thiếu ảnh dev300 -> chạy lại notebook 10 mục 4–6'
assert os.path.exists(COVERAGE), 'Thiếu ocr_coverage.json -> chạy notebook 10 mục 7'
!python -m src.eval.run_baseline --selftest 2>&1 | tail -1
!python -m src.eval.select_ocr_config --selftest

## 3. Chạy 5 run trên dev300 (≈ 30–40 phút)

Cùng script `run_baseline.py` với B1/B2. Run OCR thêm `--mode ocr --ocr_backend … --ocr_template …`; giá trị `max_chars 600` và `min_conf 0` để mặc định theo luật. Run nào đã có `metrics.json` thì bỏ qua, nên Colab có ngắt thì chỉ cần chạy lại cell.

Mỗi run nạp lại model (khoảng 1 phút), sau đó chạy 300 câu. Run có OCR chậm hơn một chút, vì prompt dài hơn.

*Kỳ vọng: 5 lượt `== KẾT QUẢ: EM=…`. Mỗi run OCR in thêm dòng `OCR: … % mẫu có chữ`.*

In [ ]:
if os.path.exists(f'{EXP_ROOT}/{B1}/metrics.json'):
    print(f'⏭ {B1}: đã có -> bỏ qua')
else:
    !python -u -m src.eval.run_baseline --dataset {DS} --data_file {DEV} --run_name {B1}

for (backend, tpl), run_name in RUNS.items():
    if os.path.exists(f'{EXP_ROOT}/{run_name}/metrics.json'):
        print(f'⏭ {run_name}: đã có -> bỏ qua'); continue
    print(f'\n▶ {backend} · {tpl}')
    !python -u -m src.eval.run_baseline --dataset {DS} --data_file {DEV} --run_name {run_name} --mode ocr --ocr_backend {backend} --ocr_template {tpl}

## 4. Áp dụng luật chọn cấu hình (đặt trước, `docs/W08/README.md`)

`select_ocr_config.py` làm đúng 3 luật:
1. **Backend:** so EM của 2 backend (template v1) bằng McNemar. p < 0,05 thì chọn backend EM cao hơn; hoà thì xét answer-in-OCR, rồi đến tốc độ.
2. **Template:** chỉ chọn v2 khi v2 tốt hơn v1 **có ý nghĩa thống kê**; còn lại giữ v1.
3. **max_chars:** chỉ cảnh báo nếu việc cắt 600 ký tự làm mất > 5 điểm answer-in-OCR.

In kèm (không thuộc luật): mỗi run OCR so với B1 zero-shot, để xem **OCR có giúp B1 trên dev không**.

*Kỳ vọng: 3 dòng `Luật 1/2/3 …`, bảng 4 dòng so với B1, dòng `➜ CHỐT: backend=… | template=…`, file `ocr_selection.json`.*

In [ ]:
args = ' '.join(f'--run {b}:{t}={EXP_ROOT}/{r}' for (b, t), r in RUNS.items())
!python -m src.eval.select_ocr_config --b1 {EXP_ROOT}/{B1} {args} --coverage {COVERAGE} --out {OUT_DIR}/ocr_selection.json

## 5. ⏸ DỪNG

Gửi Claude **toàn bộ output mục 3 (các dòng `== KẾT QUẢ`) và mục 4**. Sau đó **Runtime → Disconnect and delete runtime**.

Cấu hình in ở dòng `➜ CHỐT` được **đóng băng** cho các bước sau: Phần B của notebook 10 (OCR test2000 + tập con train), ablation B1+OCR / B2+OCR trên test2000, và train B3. Notebook này **không chạy gì trên test**.